## Cleaning up a messy dataset

Source:

Ibrahim Salami: I Cleaned a Messy CSV File Using Pandas. towards data science, 2025-11-26.

https://towardsdatascience.com/i-cleaned-a-messy-csv-file-using-pandas-heres-the-exact-process-i-follow-every-time/


![](img/clean-up.jpg)

---

### The dataset

In this notebook, we are going to use a dataset from [kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows?resource=download), namely the **Netflix_Movies_and_TV_Shows** dataset.

Netflix is one of the most popular media and video streaming platforms. They have over 8000 movies or tv shows available on their platform, as of mid-2021, they have over 200M Subscribers globally. This tabular dataset consists of listings of all the movies and tv shows available on Netflix, along with details such as cast, directors, ratings, release year, duration, etc.

---

### Overview: Standard cleaning workflow


The standard cleaning workflow is adapted from:

Ibrahim Salami: I Cleaned a Messy CSV File Using Pandas. towards data science, 2025-11-26.

https://towardsdatascience.com/i-cleaned-a-messy-csv-file-using-pandas-heres-the-exact-process-i-follow-every-time/



The standard cleaning workflow consists of 5 simple stages.

1.    Load
1.    Inspect
1.    Clean
1.    Value Standardization
1.    Export

In [ ]:
import pandas as pd
pd.options.display.max_rows = 100
pd.options.display.width = 800

---

#### Load

There are some things to keep in mind before loading your dataset. However, this is an optional step, and we probably wouldn’t encounter most of these issues in our dataset. But it doesn’t hurt to know these things. Here are some key things to consider while loading.

**Encoding issues** (utf-8, latin-1): 
Encoding defines how characters are stored as bytes in the file. Python and Pandas usually default to **UTF-8**, which handles most modern text and special characters globally. However, if the file was created in an older system or a non-English environment, it might use a different encoding, most commonly **Latin-1**.

So if you try to read a Latin-1 file with UTF-8, Pandas will encounter bytes it doesn’t recognise as valid UTF-8 sequences. You’ll typically see a `UnicodeDecodeError` when you try to read a CSV with encoding issues.

If perhaps the default load fails, you could try to specify a different encoding:

In [ ]:
# First attempt (the default)
try:
    df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv')
except UnicodeDecodeError:
    print("UnicodeDecodeError encountered. Trying a different encoding ...")
# Second attempt with a common alternative
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv', encoding='latin-1')

**Wrong delimiters**: 
CSV stands for “Comma Separated Values,” but in reality, many files use other characters as separators, like semicolons (common in Europe), tabs, or even pipes (|). Pandas typically defaults to the comma (,).

So, if your file uses a semicolon (;) but you load it with the default comma delimiter, Pandas will treat the entire row as a single column. The result would be a DataFrame with a single column containing entire lines of data, making it impossible to work with.

The fix is pretty simple. You can try checking the raw file (opening it in a text editor like VS Code or Notepad++ is best) to see what character separates the values. Then, pass that character to the sep argument like so

In [ ]:
# If the file uses semicolons
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv', sep=';')

# If the file uses tabs (TSV)
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv', sep='\t')

**Columns that import incorrectly**: 
Sometimes, Pandas guesses the data type for a column based on the first few rows, but later rows contain unexpected data (e.g., text mixed into a column that started with numbers).

For instance, Pandas may correctly identify 0.1, 0.2, 0.3 as floats, but if row 100 contains the value N/A, Pandas might force the entire column into an object (string) type to accommodate the mixed values. This sucks because you lose the ability to perform fast, vectorised numeric operations on that column until you clean up the bad values.

To fix this, I use the dtype argument to tell Pandas what data type a column should be explicitly. This prevents silent type casting.

In [ ]:
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv', dtype={'release_year': int})

**Reading the first few rows**: 
You could save time by checking the first few rows directly during the loading process using the nrows parameter. This is great, especially when you’re working with large datasets, as it allows you to test encoding and delimiters without loading the entire 10 GB file.

In [ ]:
# Load only the first 50 rows to confirm encoding and delimiter
print(df.head(50))

Once you’ve confirmed the arguments are correct, you can load the full file.

Let’s load the Employee dataset. We don’t expect to see any issues here.

In [ ]:
df = pd.read_csv('datasets/Netflix Streaming Data/Netflix Streaming Data.csv')
df.shape

---

#### Inspect

**Understanding the Boundaries**:
I always start with a visual check. I use `df.head()` and `df.tail()` to see the first and last five rows. This is a quick sanity check to see if all columns look aligned and if the data visually makes sense.

In [ ]:
print (df.head())
print ('\n')
print (df.tail())

**Spotting Datatype Problems and Missingness**: 
This is the most critical method. It tells me the column names, the data types (Dtype), and the exact number of non-null values.


In [ ]:
df.info()

Pandas has special ways of dealing with missing data depending on the data type. As you may have already noticed, certain fields in a CSV file show up as `NaN` (Not a Number) in a Pandas DataFrame. 

To filter and count the number of missing/not missing values in a dataset, we can use the special `.isna()` and `.notna()` methods on a DataFrame or Series object.

Keep in mind that the `.count()` method always excludes NaN values, so we can count the number of available values in each column. The total number of rows in each column (`size`) can be used to find the percentage of not blank data in every column.

The `.isna()` and `.notna()` methods return True/False pairs for each row, which we can use to filter the DataFrame for any rows that have information in a given column. 

In [ ]:
print(df['director'].count())
print(df['director'].size)
print(df['director'].notna().sum())
print(df['director'].isna().sum())
print(df['director'].notna().sum()/df['director'].size)

**Data Integrity Check: Duplicates and Unique Counts** 

- Checking for duplicate rows.

In [ ]:
print (df.shape)    
print ("Duplicates in description column: ", df.duplicated(subset=["description"], keep=False).sum())
print ("Number of first duplicates: ", df.duplicated(subset=["description"], keep="first").sum())
print (df[["title", "type", "cast", "description"]][df.duplicated(subset=["description"], keep=False)].sort_values(by="description"))

Oops! What a mess! We have to remove 32 duplicate lines. 

In [ ]:
df_dedup = df.drop_duplicates(subset=["description"], keep="first").reset_index(drop=True)
print (df_dedup.shape)

This is perfect! It means we no longer have rows for identical movies cluttering up our dataset.

- Checking Unique Values (`df.nunique()`): Count number of distinct elements per column (default). I use this to understand the diversity within each column. Low counts in categorical columns are fine, but I look for columns that should be unique but aren’t, or columns that have too many unique values, suggesting typos.

In [ ]:
df_dedup.nunique()

-
    - `show_id` has 8775 unique values. This is perfect.
    - `type` has 2 unique values, namely _TV Show_ and _Movie_. This is perfect.
    - `title` has 8775 unique values. This is perfect.
    - `director` has 4527 unique values. To my opinion, this is a really large number.
    - `cast` has 7686 unique values. That needs inspection.
    - `country` has 747 unique values. This large number may result from the fact that we have different combinations of countries in this column.
    - `date_added` has 1765 unique values. Ok.
    - `release_year` has 74 unique values. We may conclude that Netflix also has rather old movies.
    - `rating` has 17 unique values. That needs inspection.
    - `duration` has 220 unique values. This is a funny column.
    - `listed_in` has 514 unique values. Very high number for a categorical column. That needs inspection.
    - `description` has 8775 unique values.


**Catching Odd and Impossible Values**: 
I use `df.describe()` to get a statistical summary of all my numerical columns. This is the place where truly impossible values—the “red flags”—show up instantly. I mostly focus on the min and max rows.

In [ ]:
df_dedup.describe()

---

#### Clean

**The Consistency Rule—Standardising Column Names and Setting the Index**:
Before I do any serious data manipulation, I enforce strict consistency on column names. Why? Because typing df['Show ID '] accidentally instead of df['show_id'] is a silent, frustrating error. Once the names are clean, I set the index.

My golden rule is snake_case and lowercase everywhere, and ID columns should be the index.

I use a simple command to strip whitespace, replace spaces with underscores, and convert everything to lowercase.

In [ ]:
# The Standardization Command
df_dedup.columns = df.columns.str.lower().str.replace(' ', '_').str.strip()
df_dedup.columns

**Set index** (optional): We might move on to set `show_id` as an index.

In [ ]:
# This is crucial for efficient lookups and clean merges later.
df_dedup.set_index('show_id', inplace=True)

# Let’s review it real quick
df.head()

We revert the last step.

In [ ]:
df_dedup = df_dedup.reset_index()
df_dedup.head()

**Handling Missing Values**: 
Finally, we address the gaps revealed by `df.info()`:

In [ ]:
df_dedup.info()

We remove rows with missing `cast` values. Thus, we get rid of 820 rows.

In [ ]:
# Removal: Drop rows missing values for 'cast'
df_dedup = df_dedup.dropna(subset=['cast'])
df_dedup = df_dedup.reset_index(drop=True)
print(f"Rows after dropping missing cast values: {len(df_dedup)}\n")
df_dedup.info()

---

#### Value Standardization

The DataFrame now has the right structure, but the values inside are still dirty. This step is about consistency. If “IT,” “i.t,” and “Info. Tech” all mean the same department, we need to force them into a single, clean value (“IT”). This prevents errors in grouping, filtering, and any statistical analysis based on categories.

**Rating Values—What a mess!** 
Let's create a set of all unique rating values to see what ratings are present in the dataset.


In [ ]:
# Create a set of unique rating values
rating_set = set(df_dedup['rating'].unique())
print(f"Unique ratings ({len(rating_set)}):")
print(rating_set)


In [ ]:
row = df_dedup[df_dedup['rating'] == '66 min']
print(row)

There certainly is a problem with the `rating` column, as the example shows: `rating` and `duration` seem to be mixed up. We don't care for a solution here, as we are not interested in the ratings for the moment.

**Converting Date Columns—The date_added Fix**: 
The `date_added` column is usually read in as a string (object) type, which makes time-series analysis impossible. This means we have to convert it to a proper Pandas datetime object. 
`pd.to_datetime()` is the core function. we use `errors='coerce'` as a safety net; if Pandas can’t parse a date, it converts that value to NaT (Not a Time), which is a clean null value, preventing the whole operation from crashing.

In [ ]:
print(df_dedup['date_added'].head())
# Convert the date_added column to datetime objects
df_dedup['date_added'] = pd.to_datetime(df_dedup['date_added'], format='%B %d, %Y', errors='coerce')
df_dedup['date_added'].head()

Following format codes apply:

- %B = Full month name (September)
- %b = Abbreviated month name (Sep)
- %d = Day of month (01-31)
- %Y = 4-digit year (2001)
- %y = 2-digit year (01)
- %m = Month as number (09)

**Remove illegal characters**: Some characters may be illegal in text strings.

Find '$$' in `cast` column.

In [ ]:
df_dedup[df_dedup['cast'].str.contains('$$', na=False, regex=False)]

Replace by 'ss'.

In [ ]:
df_dedup['cast'] = df_dedup['cast'].str.replace('$$', 'ss', regex=False)

---

#### Export

Before closing the notebook, we perform one last audit to ensure everything is perfect.

**The Final Data Quality Check**: 
This is quick. I re-run the two most critical inspection methods to confirm that all my cleaning commands actually worked:

- `df.info()`: I confirm there are no more missing values in the critical columns (age, salary) and that the data types are correct (phone is a string, join_date is datetime).
- `df.describe()`: I ensure the statistical summary shows plausible numbers. The Phone column should now be absent from this output (since it’s a string), and Age and Salary should have logical minimum and maximum values.
If these checks pass, I know the data is reliable.

In [ ]:
df_dedup.info()

In [ ]:
df_dedup.describe(include='all')

**Exporting the Clean Dataset**: 
The final step is to save this cleaned version of the data. 

In [ ]:
# Exporting the clean DataFrame 
df_dedup.to_pickle('datasets/Netflix Streaming Data/Netflix Streaming Data-Cleaned.pkl')

By exporting with a clear, new filename, you officially mark the end of the cleaning phase and provide a clean slate for the next phase of the project.